# Cogni Unsloth QLoRA Training (Google Colab)

This notebook reproduces the full training process:

1. Install Unsloth and dependencies
2. Download/load base model
3. Load `datasets/training`
4. Train with QLoRA
5. Save adapters
6. Export merged model

Training only starts when you run the trainer cell.

In [ ]:
# Colab setup
!nvidia-smi

In [ ]:
# Install dependencies (Colab)
!pip -q install --upgrade pip
!pip -q install "unsloth[colab-new]" datasets trl peft bitsandbytes accelerate transformers

In [ ]:
# Optional: if your training data is in Google Drive, mount it
# from google.colab import drive
# Uncomment next line if needed
# drive.mount('/content/drive')


In [ ]:
# If needed, clone the repo so /content/cogni-slm/datasets/training exists
# !git clone https://github.com/<your-org-or-user>/Cogni-SLM.git /content/cogni-slm

In [ ]:
import json
from pathlib import Path

import torch
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth import FastLanguageModel

from datasets import load_from_disk


In [ ]:
# Config
MODEL_ID = "Qwen/Qwen3-1.7B-Instruct"
DATASET_PATH = Path("/content/cogni-slm/datasets/training")  # adjust if needed
OUTPUT_DIR = Path("/content/cogni-slm/models/unsloth_qlora")
ADAPTER_DIR = OUTPUT_DIR / "adapters"
MERGED_DIR = OUTPUT_DIR / "merged"
RUN_METADATA = OUTPUT_DIR / "run_metadata.json"

MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True

LORA_R = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.0

NUM_EPOCHS = 1.0
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.0
WARMUP_RATIO = 0.03
LR_SCHEDULER = "cosine"

PER_DEVICE_TRAIN_BATCH_SIZE = 2
PER_DEVICE_EVAL_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8

EVAL_STEPS = 50
SAVE_STEPS = 50
LOGGING_STEPS = 10
SAVE_TOTAL_LIMIT = 3
SEED = 42

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Validate training dataset path and load splits
if not DATASET_PATH.exists():
    raise FileNotFoundError(f"Training dataset not found: {DATASET_PATH}")

dataset_dict = load_from_disk(str(DATASET_PATH))
print(dataset_dict)
for split in dataset_dict.keys():
    print(split, len(dataset_dict[split]), dataset_dict[split].column_names)

In [ ]:
# Convert prompt/essay/score to a text field expected by SFTTrainer
def to_sft_text(example):
    prompt = str(example["prompt"]).strip()
    essay = str(example["essay"]).strip()
    score = float(example["score"])
    return {
        "text": (
            "You are an AP Language instructor. Evaluate the argument quality and score.\n\n"
            f"Prompt:\n{prompt}\n\n"
            f"Essay:\n{essay}\n\n"
            "Target:\n"
            f"Final score: {score:.4f}\n"
        )
    }

train_dataset = dataset_dict["train"].map(to_sft_text, desc="Format train text")
if "validation" in dataset_dict:
    eval_dataset = dataset_dict["validation"].map(to_sft_text, desc="Format eval text")
else:
    eval_dataset = None

print("train rows:", len(train_dataset))
print("eval rows:", len(eval_dataset) if eval_dataset is not None else 0)


In [ ]:
# Load base model + tokenizer (downloads base model if not already cached)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

print("Model and tokenizer loaded.")

In [ ]:
# Training arguments
bf16_available = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
fp16_available = torch.cuda.is_available() and not bf16_available

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type=LR_SCHEDULER,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    evaluation_strategy="steps" if eval_dataset is not None else "no",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    logging_strategy="steps",
    logging_steps=LOGGING_STEPS,
    report_to=[],
    seed=SEED,
    fp16=fp16_available,
    bf16=bf16_available,
    remove_unused_columns=False,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    args=training_args,
)

print("Trainer initialized. Training has not started yet.")

In [ ]:
# Start training (run this cell to train)
train_result = trainer.train()
print(train_result)

In [ ]:
# Save adapters
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
trainer.model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
print(f"Saved adapters to: {ADAPTER_DIR}")

In [ ]:
# Export merged model (16-bit)
MERGED_DIR.mkdir(parents=True, exist_ok=True)
trainer.model.save_pretrained_merged(
    str(MERGED_DIR),
    tokenizer,
    save_method="merged_16bit",
)
print(f"Saved merged model to: {MERGED_DIR}")

In [ ]:
# Save run metadata
metadata = {
    "model_id": MODEL_ID,
    "dataset_path": str(DATASET_PATH),
    "output_dir": str(OUTPUT_DIR),
    "adapter_dir": str(ADAPTER_DIR),
    "merged_dir": str(MERGED_DIR),
    "max_seq_length": MAX_SEQ_LENGTH,
    "load_in_4bit": LOAD_IN_4BIT,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "num_train_epochs": NUM_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "train_rows": len(train_dataset),
    "eval_rows": len(eval_dataset) if eval_dataset is not None else 0,
}
RUN_METADATA.parent.mkdir(parents=True, exist_ok=True)
RUN_METADATA.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print(f"Wrote metadata: {RUN_METADATA}")

## Artifacts

After running all training/export cells, you should have:

- Adapter weights in `models/unsloth_qlora/adapters/`
- Merged model in `models/unsloth_qlora/merged/`
- Run metadata in `models/unsloth_qlora/run_metadata.json`